In [1]:
# First install the extendedjetpy package with: python setup.py install

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import JetCode # CHANGE TO extendedjetpy CUANDO ANDE BIEN!!!!

In [2]:
#Constants

mJy = 1.0e-26
dist = 16.8*3.0856e24  # Distance to the source = 16.8 Mpc
h = 6.6260755e-27     # erg.seg
eV = 1.602176e-12     # 1 eV = 1.602176e-12 erg
a = 4.0*np.pi*dist**2  
M_BH = 6.5e9             #Black hole mass [Msun] (EHT+2018?)
R_g = 1.477e5*M_BH       #Gravitational radius (M_BH [Msun])  

In [3]:
# Define the function to fit

def model_sed(E, fB, Lj, z0):
    """Synchrotron + SSC model calculated at photon energies in the vector E.
    The available free parameters are 
    - fB (magnetic field B fraction) 
    - Lj (jet luminosity)"""
    # CHANGE TO extendedjetpy CUANDO ANDE BIEN!!!!
    fit_values = JetCode.sed_model(fB, Lj, z0, E) # The routine returns m_data_max values, as defined in global.f90
    return( fit_values[:len(E)])     # The only useful values are the first ones up to the number of points in E

In [4]:
# Define the energy range for the SED

E_min = 1.e-7 * eV #Aprox 20 MHz
E_max = 1.e10 * eV
n_points = 80
E = np.geomspace(E_min, E_max, n_points)

In [5]:
# Make a list with the values of fB 
fBs = [1e-3, 1e-2, 1e-1] 
#eta_names = ['1e-4', '1e-3', '1e-2', '1e-1'] 
print(fBs)

[0.001, 0.01, 0.1]


In [ ]:
# Calculate the SED for each fB (fixing default parameters)
Lj = 1e43
z0 = 10.0*R_g
SEDs = []
for fB in fBs:
    print(fB)
    SED = model_sed(E, fB, z0, Lj)
    SEDs.append(SED)

In [ ]:
# Voy a graficar datos y modelo
fig, ax = plt.subplots(1)

# Defino una paleta discreta
n = len(fBs)
color_vector = np.linspace(0.2, 0.9, n)
colors = plt.cm.copper_r(color_vector) # define a palette avoiding extreme colors
# Genero una colorbar discreta para n puntos
c = np.arange(1, n+1) 
cmap = plt.cm.get_cmap('copper_r', n)
norm = matplotlib.colors.BoundaryNorm(np.arange(n+1)+0.5,n)
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

# Definiciones estéticas
nice_fonts = {
        # Use LaTeX to write all text
        "text.usetex": True,
        "font.family": 'serif',
        # Use 10pt font in plots, to match 10pt font in document
        "axes.labelsize": 18,
        "font.size": 18,
        "axes.linewidth":1,
        "axes.titlesize":16,
        # Make the legend/label fonts a little smaller
        "legend.fontsize": 12,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "xtick.major.size": 5,     # major tick size in points
        "xtick.minor.size": 5,       # minor tick size in points
        "xtick.major.width": 1.4,     # major tick width in points
        "xtick.minor.width": 1.4,     # minor tick width in points
}
matplotlib.rcParams.update(nice_fonts)

# Set labels
plt.title('')
plt.setp(ax, xlabel=r"$E$ [eV]")
plt.setp(ax, ylabel=r"$E \times L_E$ [erg s$^{-1}$]")

# Set plot scales
xmin, xmax = E_min/eV, E_max/eV
ymin, ymax = 1e34, 1e42
plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)

# Grafico las funciones individuales y su suma usando una paleta discreta
for i in range(n):
    plt.plot(E/eV, E*SEDs[i]*a, ls='-', color=colors[i], linewidth=2, label='', zorder=1)

plt.legend(loc='best')
ax.loglog() 

cbar = fig.colorbar(sm, ticks=c, label=r"$f_B$", pad=0.015)
cbar.ax.set_yticklabels(fBs)  # vertically oriented colorbar

# Set xticks
num = int(np.log10(xmax/xmin))
x_major = matplotlib.ticker.LogLocator(base = 10.0, numticks = num//3+1)
ax.xaxis.set_major_locator(x_major)
x_minor = matplotlib.ticker.LogLocator(base = 10.0, numticks = num)
ax.xaxis.set_minor_locator(x_minor)
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())

plt.show()

In [ ]:
#fig.tight_layout(pad=0)
#fig.savefig('SEDs_etas.pdf', bbox_inches='tight', pad_inches = 0.01)
#AJUSTO A MANO EL RECORTE
fig.subplots_adjust(bottom=0.16, top=0.97, left=0.14, right=0.91)
fig.savefig('SEDs_fBs.pdf')

In [ ]:
print(fBs[0])
Lj = 1e43
z0 = 10.0*R_g
SED = model_sed(E, fBs[0], z0, Lj)